# H1 check: stratospheric N₂O vs surface N₂O beyond latitude

**Question:** is lower-stratospheric N₂O above high-emission sites higher than expected for their latitude?

**Method**
1. For every site and pressure level, compute the 2005–2021 mean N₂O mixing ratio. Monthly means are formed from the daily values (weighted by the number of profiles in the third column); months with fewer than `MIN_MONTHLY_N` profiles are dropped, as in the manuscript; the site mean is the mean of the 12 calendar-month means, so uneven seasonal sampling does not bias it. Every site uses the same period, so the instrument drift affects all sites in the same way and drops out of the comparison.
2. Fit a smooth curve (LOWESS) of the site means against signed latitude, using every site with MLS files, including those without flux data. Signed latitude keeps the higher SH values at the upper levels out of the residuals.
3. Each site's residual is how far its stratospheric N₂O sits above or below what its latitude predicts. Test whether the residuals are related to surface N₂O (EDGAR inventory and chamber fluxes from `data_pca4.csv`) with Spearman rank correlation (not affected by the skewed fluxes); p-values are analytic and from a permutation test.
4. Repeat with other LOWESS smoothing settings (`FRAC_SENSITIVITY`) so the result does not hinge on one choice.

**Input files:** `<DATA_DIR>/<level>/n2oavg_<location>_<level>.csv`, no header, comma-separated: date as YYYYDDD, N₂O (ppbv), number of profiles.

**Reading the result:** `rho_before_lat` is the apparent relation before latitude is removed. `rho_resid` is the test. Small values with p > 0.05 at every level, and similar values in the sensitivity columns, support H1. With ~32 sites only |ρ| of about 0.5 or more is reliably detectable.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)

## Settings
Set `FLUX_CSV` to the location of `data_pca4.csv`.

In [ ]:
DATA_DIR = Path(r"C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o")
FLUX_CSV = Path(r"C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\pca_v6\data_pca4.csv")          # <- set this
OUT_DIR = DATA_DIR.parent / "h1_test"

LEVELS = [68, 46, 32, 22]
YEARS = (2005, 2021)
MIN_MONTHLY_N = 5          # months with fewer profiles are dropped
LOWESS_FRAC = 0.4          # share of sites in each local fit (main result)
FRAC_SENSITIVITY = [0.3, 0.5, 0.6]
N_PERM = 10000
SEED = 1

# Sites that have MLS files but no flux data. They are used only for the
# latitude fit. Keys must match the <location> part of the file names; the
# notebook lists any MLS site it cannot find a latitude for.
EXTRA_LAT = {
    "kongo": 1.35,
    "tarapoto": -6.47,
    "mexico": 19.3,
}

## Functions

In [ ]:
def read_mls(path):
    df = pd.read_csv(path, header=None).iloc[:, :3]
    df = df.apply(pd.to_numeric, errors="coerce").dropna(subset=[0, 1])
    df.columns = ["yyyyddd", "n2o", "n"][: df.shape[1]]
    if "n" not in df:
        df["n"] = 1
    df["date"] = pd.to_datetime(df["yyyyddd"].astype(int).astype(str), format="%Y%j")
    return df


def site_mean(df):
    d = df[df["date"].dt.year.between(*YEARS)].copy()
    d["yr"], d["mo"] = d["date"].dt.year, d["date"].dt.month
    d["wsum"] = d["n2o"] * d["n"]
    m = d.groupby(["yr", "mo"])[["wsum", "n"]].sum()
    m = m[m["n"] >= MIN_MONTHLY_N]
    clim = (m["wsum"] / m["n"]).groupby(level="mo").mean()
    return clim.mean() if len(clim) == 12 else np.nan


def load_mls():
    pat = re.compile(r"^n2oavg_(.+)_(\d+)$")
    out = {}
    for lev in LEVELS:
        files = sorted((DATA_DIR / str(lev)).glob("n2oavg_*.csv"))
        print(f"{lev} hPa: {len(files)} files")
        for f in files:
            mm = pat.match(f.stem)
            if mm and int(mm.group(2)) == lev:
                out.setdefault(mm.group(1).lower(), {})[f"N2O_{lev}"] = site_mean(read_mls(f))
    return pd.DataFrame.from_dict(out, orient="index")


def load_surface():
    fx = pd.read_csv(FLUX_CSV, sep=";")
    fx["loc"] = fx["loc_inname"].astype(str).str.strip().str.lower()
    return fx.groupby("loc").agg(
        name=("Asukoht", "first"),
        lat=("lat", "first"),
        flux=("N2O_wetland", "median"),
        inventory=("N2O_inventory", "mean"),
        n_flux=("N2O_wetland", "size"),
    )


def spearman_tests(x, y, rng):
    """Spearman rho, analytic p and two-sided permutation p."""
    x, y = stats.rankdata(x), stats.rankdata(y)
    r = np.corrcoef(x, y)[0, 1]
    df = len(x) - 2
    p = 2 * stats.t.sf(abs(r) * np.sqrt(df / (1 - r**2)), df)
    null = np.array([np.corrcoef(rng.permutation(x), y)[0, 1] for _ in range(N_PERM)])
    p_perm = (np.sum(np.abs(null) >= abs(r)) + 1) / (N_PERM + 1)
    return r, p, p_perm


def lat_fit(lat, y, frac):
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        return lowess(y, lat, frac=frac, return_sorted=False), f"LOWESS, frac = {LOWESS_FRAC}"
    except ImportError:
        return np.polyval(np.polyfit(lat, y, 4), lat), "4th-order polynomial (statsmodels not installed)"


def zone(lat):
    return "NH" if lat >= 30 else ("SH" if lat < -30 else "Tropics")

## Load MLS site means and surface data
Check the printed file counts and any warnings about unmatched names before going on.

In [ ]:
mls = load_mls()
surf = load_surface()
sites = mls.join(surf, how="outer")

no_mls = sites.index[sites[[f"N2O_{l}" for l in LEVELS]].isna().all(axis=1)]
if len(no_mls):
    print("Flux sites with no matching MLS file name:", list(no_mls))
sites["lat"] = sites["lat"].fillna(pd.Series(EXTRA_LAT))
no_lat = sites.index[sites["lat"].isna()]
if len(no_lat):
    print("MLS sites without latitude (add to EXTRA_LAT):", list(no_lat))
sites = sites.dropna(subset=["lat"])
sites["zone"] = sites["lat"].apply(zone)
sites.sort_values("lat").round(2)

## Test

In [ ]:
rng = np.random.default_rng(SEED)
rows, fit_name = [], None
for lev in LEVELS:
    col = f"N2O_{lev}"
    s = sites.dropna(subset=[col])
    fits = {}
    for frac in [LOWESS_FRAC] + FRAC_SENSITIVITY:
        fit, name = lat_fit(s["lat"].values, s[col].values, frac)
        fits[frac] = pd.Series(s[col].values - fit, index=s.index)
        if frac == LOWESS_FRAC:
            fit_name = name
    sites.loc[s.index, f"resid_{lev}"] = fits[LOWESS_FRAC]

    for pred in ["inventory", "flux"]:
        t = sites.dropna(subset=[col, pred])
        rho, p, p_perm = spearman_tests(t[pred].values, t[f"resid_{lev}"].values, rng)
        row = dict(level_hPa=lev, predictor=pred, n_sites=len(t),
                   rho_before_lat=stats.spearmanr(t[pred], t[col])[0],
                   rho_resid=rho, p=p, p_perm=p_perm)
        for frac in FRAC_SENSITIVITY:
            row[f"rho_frac{frac}"] = stats.spearmanr(t[pred], fits[frac][t.index])[0]
        rows.append(row)

res = pd.DataFrame(rows)
n = int(res["n_sites"].max())
r_min = np.tanh(2.8 / np.sqrt(n - 3))   # 80% power, alpha 0.05
print(f"Latitude fit: {fit_name}")
print(f"With {n} sites, only |rho| of about {r_min:.2f} or more would be detected reliably "
      "(80% power, alpha = 0.05).")
res.round(3)

In [ ]:
# p-values for the correlation before latitude is removed (the table only had them for the residuals)
res = res.drop(columns=["p_before_lat", "p_perm_before_lat"], errors="ignore")   # safe to rerun
rng_b = np.random.default_rng(SEED + 1)
before = []
for lev in LEVELS:
    col = f"N2O_{lev}"
    for pred in ["inventory", "flux"]:
        t = sites.dropna(subset=[col, pred])
        rho, p, p_perm = spearman_tests(t[pred].values, t[col].values, rng_b)
        before.append(dict(level_hPa=lev, predictor=pred, p_before_lat=p, p_perm_before_lat=p_perm))
res = res.merge(pd.DataFrame(before), on=["level_hPa", "predictor"])
res = res[["level_hPa", "predictor", "n_sites", "rho_before_lat", "p_before_lat", "p_perm_before_lat",
           "rho_resid", "p", "p_perm"] + [c for c in res.columns if c.startswith("rho_frac")]]
res.round(3)

## Save results

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
res.to_csv(OUT_DIR / "h1_results.csv", sep=";", index=False)
sites.to_csv(OUT_DIR / "h1_site_table.csv", sep=";", index_label="loc")
print(f"Saved to {OUT_DIR}")

## Figure
Stratospheric N₂O minus the latitude fit, against surface N₂O. Saved as PNG and SVG.

In [ ]:
colors = {"NH": "#3a6ea5", "Tropics": "#e08a2e", "SH": "#3c8f4f"}
fig, axes = plt.subplots(2, len(LEVELS), figsize=(3.2 * len(LEVELS), 6.4), sharey="col")
for j, lev in enumerate(LEVELS):
    for i, pred in enumerate(["inventory", "flux"]):
        ax = axes[i, j]
        t = sites.dropna(subset=[f"resid_{lev}", pred])
        for z, g in t.groupby("zone"):
            ax.scatter(g[pred], g[f"resid_{lev}"], s=22, color=colors[z], label=z)
        r = res[(res.level_hPa == lev) & (res.predictor == pred)].iloc[0]
        ax.axhline(0, color="0.6", lw=0.8)
        ax.text(0.03, 0.95, f"ρ = {r.rho_resid:.2f}, p = {r.p_perm:.2f}",
                transform=ax.transAxes, va="top", fontsize=8)
        if pred == "flux":
            ax.set_xscale("symlog", linthresh=1)
            ax.set_xticks([-1, 0, 1, 10, 100, 1000])
            ax.set_xticklabels(["-1", "0", "1", "10", "100", "1000"])
        else:
            ax.set_xscale("log")
        if i == 0:
            ax.set_title(f"{lev} hPa")
        if j == 0:
            ax.set_ylabel("N$_2$O minus latitude fit (ppbv)")
        ax.set_xlabel("EDGAR inventory" if pred == "inventory" else "Chamber flux (site median)",
                      fontsize=8)
axes[0, 0].legend(fontsize=7, frameon=False)
fig.tight_layout()
for ext in ["png", "svg"]:
    fig.savefig(OUT_DIR / f"h1_residuals.{ext}", dpi=200)
plt.show()

In [ ]:
for lev in LEVELS:
    t = sites.dropna(subset=[f"resid_{lev}", "inventory"])
    r_inv, p_inv = stats.pearsonr(np.log10(t["inventory"]), t[f"resid_{lev}"])
    f = sites.dropna(subset=[f"resid_{lev}", "flux"])
    slog = np.sign(f["flux"]) * np.log10(1 + np.abs(f["flux"]))
    r_fx, p_fx = stats.pearsonr(slog, f[f"resid_{lev}"])
    print(f"{lev} hPa: log10(EDGAR) r = {r_inv:+.2f} (p = {p_inv:.2f}); "
          f"signed-log flux r = {r_fx:+.2f} (p = {p_fx:.2f})")

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess
fig, axes = plt.subplots(1, len(LEVELS), figsize=(14, 3.5))
for ax, lev in zip(axes, LEVELS):
    s = sites.dropna(subset=[f"N2O_{lev}"]).sort_values("lat")
    for z, g in s.groupby("zone"):
        ax.scatter(g["lat"], g[f"N2O_{lev}"], s=15, color=colors[z])
    for frac in [LOWESS_FRAC] + FRAC_SENSITIVITY:
        ax.plot(s["lat"], lowess(s[f"N2O_{lev}"], s["lat"], frac=frac, return_sorted=False), lw=1, label=frac)
    ax.set_title(f"{lev} hPa")
axes[0].legend(title="frac", fontsize=7)

In [ ]:
# Within-zone check: latitude removed separately in each zone with a quadratic in |latitude|
# (plus an SH offset in the extratropics). Reusing the all-sites LOWESS residuals inside a zone
# gave spurious correlations in a test with dummy data, so they are not used here.
rng = np.random.default_rng(SEED)
 
 
def zone_residuals(g, col):
    g = g.dropna(subset=[col])
    a = g["lat"].abs().values
    cols = [np.ones(len(g)), a, a**2]
    sh = (g["lat"] < -30).astype(float).values
    if 0 < sh.sum() < len(g):
        cols.append(sh)
    X = np.column_stack(cols)
    y = g[col].values
    return pd.Series(y - X @ np.linalg.lstsq(X, y, rcond=None)[0], index=g.index)

In [ ]:
groups = {"Tropics": sites[sites["zone"] == "Tropics"],
          "Extratropics": sites[sites["zone"] != "Tropics"]}
rows = []
for gname, g in groups.items():
    for lev in LEVELS:
        res = zone_residuals(g, f"N2O_{lev}")
        for pred in ["inventory", "flux"]:
            t = g.loc[res.index].dropna(subset=[pred])
            rho, p, p_perm = spearman_tests(t[pred].values, res.loc[t.index].values, rng)
            rows.append(dict(group=gname, level_hPa=lev, predictor=pred, n_sites=len(t),
                             rho=rho, p=p, p_perm=p_perm))
by_zone = pd.DataFrame(rows)
for gname, n in by_zone.groupby("group")["n_sites"].max().items():
    print(f"{gname}: {n} sites; |rho| of about {np.tanh(2.8 / np.sqrt(n - 3)):.2f} needed for 80% power")
by_zone.to_csv(OUT_DIR / "h1_results_by_zone.csv", sep=";", index=False)
by_zone.round(3)

In [ ]:
# Figure for the within-zone check: residuals from the zone-wise latitude fits against surface N2O.
# Run after the within-zone cell (uses zone_residuals, groups and by_zone from it).
colors = {"NH": "#3a6ea5", "Tropics": "#e08a2e", "SH": "#3c8f4f"}
label_colors = {"Tropics": "#c0661a", "Extratropics": "0.25"}

fig, axes = plt.subplots(2, len(LEVELS), figsize=(3.3 * len(LEVELS), 6.8), sharey="col")
for j, lev in enumerate(LEVELS):
    res = pd.concat([zone_residuals(g, f"N2O_{lev}") for g in groups.values()])
    for i, pred in enumerate(["inventory", "flux"]):
        ax = axes[i, j]
        t = sites.loc[res.index].assign(resid=res).dropna(subset=[pred])
        for z in ["NH", "Tropics", "SH"]:
            g = t[t["zone"] == z]
            ax.scatter(g[pred], g["resid"], s=22, color=colors[z], label=z, zorder=3)
        ax.axhline(0, color="0.6", lw=0.8, zorder=1)
        for k, gname in enumerate(groups):
            r = by_zone[(by_zone.group == gname) & (by_zone.level_hPa == lev)
                        & (by_zone.predictor == pred)].iloc[0]
            ax.text(0.03, 0.97 - 0.075 * k, f"{gname}: ρ = {r.rho:+.2f}, p = {r.p_perm:.2f}",
                    transform=ax.transAxes, va="top", fontsize=7.5, color=label_colors[gname])
        if pred == "flux":
            ax.set_xscale("symlog", linthresh=1)
            ax.set_xticks([-1, 0, 1, 10, 100, 1000])
            ax.set_xticklabels(["-1", "0", "1", "10", "100", "1000"])
        else:
            ax.set_xscale("log")
        ax.margins(y=0.25)
        if i == 0:
            ax.set_title(f"{lev} hPa")
        if j == 0:
            ax.set_ylabel("N$_2$O minus zone latitude fit (ppbv)")
        ax.set_xlabel("EDGAR inventory" if pred == "inventory" else "Chamber flux (site median)",
                      fontsize=8)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=9, frameon=False)
fig.tight_layout(rect=[0, 0.04, 1, 1])
for ext in ["png", "svg"]:
    fig.savefig(OUT_DIR / f"h1_residuals_by_zone.{ext}", dpi=200)
plt.show()